In [20]:
import pandas as pd
import tracemalloc
pd.set_option("display.max.columns", None)


In [21]:
def trophies(row):
    if row["outcome"] == "winner.tag":
        return row["winner.startingTrophies"]
    else:
        return row["loser.startingTrophies"]

def rename_outcome(row):
    if row["outcome"] == "winner.tag":
        return "win"
    return "loss"



In [ ]:
# 2.5GB peak without del
# 1.7GB peak with del

# tracemalloc.start()

# load the first batch to define dimensions of the df
df = pd.read_csv("data/BattlesStaging_01042021_WL_tagged/BattlesStaging_01042021_WL_tagged.csv")

metadata = ["winner.clan.tag", "winner.clan.badgeId", "winner.kingTowerHitPoints", "winner.princessTowersHitPoints", "loser.clan.tag", "loser.clan.badgeId", "loser.kingTowerHitPoints", "loser.princessTowersHitPoints", "tournamentTag"]
card_data = []
for player in ["winner", "loser"]:
    card_data.append(f"{player}.totalcard.level")
    card_data.append(f"{player}.troop.count")
    card_data.append(f"{player}.structure.count")
    card_data.append(f"{player}.spell.count")
    card_data.append(f"{player}.common.count")
    card_data.append(f"{player}.rare.count")
    card_data.append(f"{player}.epic.count")
    card_data.append(f"{player}.legendary.count")
    card_data.append(f"{player}.elixir.average")
    card_data.append(f"{player}.cards.list")
    for i in range(1,9):
        card_data.append(f"{player}.card{i}.id")
        card_data.append(f"{player}.card{i}.level")

combined_metadata = metadata + card_data
cleaned = df.drop(columns=combined_metadata)

del df

temp = cleaned.melt(id_vars=["battleTime","winner.startingTrophies", "loser.startingTrophies"], value_vars=["winner.tag", "loser.tag"], var_name="outcome", value_name="user")
all_players = temp.dropna()
print(f"removed {temp.size - all_players.size} rows that contained NA")

del temp

all_players["trophies"] = all_players.apply(trophies, axis=1)

all_players["outcome"] = all_players.apply(rename_outcome, axis=1)
all_players = all_players.drop(columns=["winner.startingTrophies", "loser.startingTrophies"])
# all_players = all_players.sort_values(by=["battleTime"])

# current, peak = tracemalloc.get_traced_memory()
# print(f"peak: {peak / 10**6:.2f}mb")

removed 0 rows that contained NA


In [24]:
def add_to_all_players(path: str, free_vars=True):
    global all_players
    df = pd.read_csv(path)
    print(f"loading {df.size} new rows")

    metadata = ["winner.clan.tag", "winner.clan.badgeId", "winner.kingTowerHitPoints", "winner.princessTowersHitPoints", "loser.clan.tag", "loser.clan.badgeId", "loser.kingTowerHitPoints", "loser.princessTowersHitPoints", "tournamentTag"]
    card_data = []
    for player in ["winner", "loser"]:
        card_data.append(f"{player}.totalcard.level")
        card_data.append(f"{player}.troop.count")
        card_data.append(f"{player}.structure.count")
        card_data.append(f"{player}.spell.count")
        card_data.append(f"{player}.common.count")
        card_data.append(f"{player}.rare.count")
        card_data.append(f"{player}.epic.count")
        card_data.append(f"{player}.legendary.count")
        card_data.append(f"{player}.elixir.average")
        card_data.append(f"{player}.cards.list")
        for i in range(1,9):
            card_data.append(f"{player}.card{i}.id")
            card_data.append(f"{player}.card{i}.level")

    combined_metadata = metadata + card_data
    cleaned = df.drop(columns=combined_metadata)

    if free_vars:
        del df

    temp = cleaned.melt(id_vars=["battleTime","winner.startingTrophies", "loser.startingTrophies"], value_vars=["winner.tag", "loser.tag"], var_name="outcome", value_name="user")

    if free_vars:
        del cleaned
    
    batch = temp.dropna()
    print(f"removed {temp.size - batch.size} rows that contained NA")

    if free_vars:
        del temp

    batch["trophies"] = batch.apply(trophies, axis=1)
    batch["outcome"] = batch.apply(rename_outcome, axis=1)
    batch = batch.drop(columns=["winner.startingTrophies", "loser.startingTrophies"])

    all_players = pd.concat([all_players, batch])

    if free_vars:
        del batch


In [39]:
# 1.6GB peak with del
# 2.1GB peak without del

# tracemalloc.start()

print(f"starting with {all_players.size} rows")
# add_to_all_players("data/BattlesStaging_01032021_WL_tagged/BattlesStaging_01032021_WL_tagged.csv")
# add_to_all_players("data/BattlesStaging_01022021_WL_tagged/BattlesStaging_01022021_WL_tagged.csv")
# add_to_all_players("data/BattlesStaging_01012021_WL_tagged/BattlesStaging_01012021_WL_tagged.csv")
# add_to_all_players("data/BattlesStaging_12312020_WL_tagged/BattlesStaging_12312020_WL_tagged.csv")
# add_to_all_players("data/BattlesStaging_12302020_WL_tagged/BattlesStaging_12302020_WL_tagged.csv")
# add_to_all_players("data/BattlesStaging_12292020_WL_tagged/BattlesStaging_12292020_WL_tagged.csv")
# add_to_all_players("data/BattlesStaging_12282020_WL_tagged/BattlesStaging_12282020_WL_tagged.csv")
# add_to_all_players("data/BattlesStaging_12272020_WL_tagged/BattlesStaging_12272020_WL_tagged.csv")
add_to_all_players("data/BattlesStaging_12072020_to_12262020_WL_tagged/battlesStaging_12072020_to_12262020_WL_tagged.csv")

# peak, current = tracemalloc.get_traced_memory()
# print(f"peak: {peak / 10**6:.2f}mb")

starting with 169419424 rows
loading 1242900966 new rows
removed 0 rows that contained NA


In [37]:
game_count_per_player = all_players.groupby('user')['trophies'].count().reset_index(name="count")
game_count_per_player.head()
# game_count_per_player['count']

,user,count
0,#200000RR,1
1,#200000YYR,3
2,#200002PLJ,2
3,#200002VGY,16
4,#200002VLY,6


In [43]:
all_players.tail()

,battleTime,outcome,user,trophies
33591913,2020-12-27 00:02:00+00:00,loss,#9PYLQ0VYQ,5191.0
33591914,2020-12-27 00:02:22+00:00,loss,#G80VR88,5340.0
33591915,2020-12-27 00:02:58+00:00,loss,#9Q9UPPP2L,4949.0
33591916,2020-12-27 00:03:32+00:00,loss,#YYGJ2JQL0,4208.0
33591917,2020-12-27 00:03:37+00:00,loss,#8CGL8VV00,4920.0


In [51]:
temp = all_players.sort_values("battleTime").reset_index(drop="index")
temp.tail()

,battleTime,outcome,user,trophies
75946769,2021-01-04 08:03:59+00:00,win,#UCG8C90,4872.0
75946770,2021-01-04 08:03:59+00:00,win,#GYULLLJ2U,3642.0
75946771,2021-01-04 08:03:59+00:00,loss,#RPCR282YQ,3625.0
75946772,2021-01-04 08:04:00+00:00,win,#YY0PQ0UGY,3969.0
75946773,2021-01-04 08:04:00+00:00,loss,#R8P82QRQU,3966.0


In [53]:
temp.to_csv("data/1207to0104.csv")